# check

> Draw a demo in a headless browser and play it: no errors, no network, the gate hides the answer, keys move each control, and the text has enough contrast.

`validate` reads the spec. `check` runs it. It writes the static page, opens it in a headless browser with playwright, and plays the demo as a learner would.
The browser is always headless, so no window opens and nothing takes the focus.

These checks run in light and in dark mode:

1. The demo draws: the host gets `data-rk-ready="1"`. Any console error, page error, or request that is not `file:`, `data:` or `about:` fails.
2. Each visible block has a box that is not empty. Each block with `after` is hidden before its gate.
3. Each text has a contrast of at least 4.5:1 against what is behind it (3:1 for large text), before and after the reveal.
4. Play: for each gate, pick the first option and lock in with ⌘↩. The verdict shows, the `rk-predict` event fires, its answer matches the Python twin, and the blocks that wait show.
5. Keys: each control gets a key press, and the state must move. The derived values must match the Python twins.

In [ ]:
#| default_exp check

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import asyncio, json, math, os, tempfile, threading
from concurrent.futures import ThreadPoolExecutor
from contextlib import contextmanager
from pathlib import Path
from fastcore.utils import *
from rupaka.spec import run, get, js_str, _gates
from rupaka.render import save

## The browser

The check never uses the Chrome that you use every day: a headless run of that app shares its crash reporter, its updater and its app registration, and many runs can crash it.
It uses, in this order: a path that you give, the path in `$FASTCDP_CHROME`, or the Chromium that `playwright install chromium` gives.
One process runs one browser at a time, and each browser closes in a `finally` block.

In [ ]:
#| export
def browser_path(path:str=None):
    "The browser to use: `path`, else `$FASTCDP_CHROME`, else None for playwright's own Chromium. Never the everyday Chrome."
    p = path or os.getenv('FASTCDP_CHROME') or None
    if p and '/Applications/Google Chrome.app' in p: raise ValueError('rupaka does not run the everyday Chrome; use playwright Chromium or $FASTCDP_CHROME')
    return p

_ONE = threading.RLock()

@contextmanager
def browser(path:str=None):
    "A headless browser for `check`; one at a time in this process, closed at the end."
    from playwright.sync_api import sync_playwright
    with _ONE, sync_playwright() as pw:
        b = pw.chromium.launch(executable_path=browser_path(path), headless=True)
        try: yield b
        finally: b.close()

The default is playwright's own Chromium, not the everyday Chrome, and a path into the everyday Chrome is refused.

In [ ]:
import os
if not os.getenv('FASTCDP_CHROME'): test_eq(browser_path(), None)
test_fail(lambda: browser_path('/Applications/Google Chrome.app/Contents/MacOS/Google Chrome'), contains='everyday Chrome')

`can_check` says if a browser starts here. The notebook runs its browser tests only when one does.

In [ ]:
#| export
def _loop_running():
    try: asyncio.get_running_loop(); return True
    except RuntimeError: return False

def in_thread(f, *a, **kw):
    "Call `f`; in a thread when an asyncio loop runs here (as in a notebook), because playwright's sync API cannot run in a loop."
    if not _loop_running(): return f(*a, **kw)
    with ThreadPoolExecutor(1) as ex: return ex.submit(f, *a, **kw).result()

def _starts():
    with browser(): return True

@functools.cache
def can_check() -> bool:
    "True when playwright and a browser are there."
    try: return in_thread(_starts)
    except Exception: return False

## Contrast

This script runs in the page. For each element with its own text, it finds the colour behind the text: the backgrounds of the element and its parents,
and for SVG text, a shape behind it in the same group (the dot of a picked wheel item, the cell of a grid). It mixes in opacity, then gives the WCAG ratio.
Disabled controls and hidden text are skipped.

In [ ]:
#| export
CONTRAST_JS = r"""() => {
  const parse = c => { let m = c && c.match(/rgba?\(([^)]+)\)/);
    if (m) { const p = m[1].split(/[\s,\/]+/).filter(Boolean).map(Number); return [p[0], p[1], p[2], p[3] ?? 1]; }
    m = c && c.match(/color\(srgb ([^)]+)\)/);
    if (m) { const p = m[1].split(/[\s\/]+/).filter(Boolean).map(Number); return [p[0] * 255, p[1] * 255, p[2] * 255, p[3] ?? 1]; }
    return null; };
  const over = (t, b) => [0, 1, 2].map(i => t[i] * t[3] + b[i] * (1 - t[3])).concat(1);
  const lum = c => { const f = v => { v /= 255; return v <= .03928 ? v / 12.92 : ((v + .055) / 1.055) ** 2.4; }; return .2126 * f(c[0]) + .7152 * f(c[1]) + .0722 * f(c[2]); };
  const ratio = (a, b) => { const x = lum(a), y = lum(b); return (Math.max(x, y) + .05) / (Math.min(x, y) + .05); };
  const base = parse(getComputedStyle(document.body).backgroundColor) || [255, 255, 255, 1];
  const bgOf = e => { const st = [];
    for (let n = e; n && n.nodeType === 1; n = n.parentElement) { const c = parse(getComputedStyle(n).backgroundColor); if (c && c[3] > 0) { st.push(c); if (c[3] >= 1) break; } }
    let col = base[3] >= 1 ? base : [255, 255, 255, 1]; for (let i = st.length - 1; i >= 0; i--) col = over(st[i], col);
    if (e instanceof SVGElement) { const r = e.getBoundingClientRect(), cx = r.x + r.width / 2, cy = r.y + r.height / 2;
      for (const s of e.parentElement.children) { if (s === e || !['circle', 'rect', 'path'].includes(s.tagName)) continue;
        const q = s.getBoundingClientRect(); if (cx < q.left || cx > q.right || cy < q.top || cy > q.bottom) continue;
        const f = parse(getComputedStyle(s).fill); if (f && f[3] > 0) col = over(f, col); } }
    return col; };
  const out = [];
  document.querySelectorAll('.rk *').forEach(e => {
    if (![...e.childNodes].some(n => n.nodeType === 3 && n.textContent.trim())) return;
    if (!e.checkVisibility({opacityProperty: true, visibilityProperty: true}) || e.closest('.rk-sr, :disabled')) return;
    const r = e.getBoundingClientRect(); if (!r.width || !r.height) return;
    const cs = getComputedStyle(e), fg0 = parse(e instanceof SVGElement ? cs.fill : cs.color); if (!fg0) return;
    let op = 1; for (let n = e; n && n.nodeType === 1; n = n.parentElement) op *= +getComputedStyle(n).opacity;
    const bg = bgOf(e), fg = over([fg0[0], fg0[1], fg0[2], fg0[3] * op], bg), px = parseFloat(cs.fontSize), big = px >= 24 || (px >= 18.66 && +cs.fontWeight >= 700);
    out.push({text: e.textContent.trim().slice(0, 40), ratio: +ratio(fg, bg).toFixed(2), need: big ? 3 : 4.5});
  });
  return out;
}"""

def contrast(pg) -> list:
    "The texts of the page whose contrast is under the WCAG need, as `(text, ratio, need)`."
    return [(x['text'], x['ratio'], x['need']) for x in pg.evaluate(CONTRAST_JS) if x['ratio'] < x['need']]

## The parts of the check

`_state` reads the state of the demo from the page. `_parity` compares each derived value with the Python twin of its fn.

In [ ]:
#| export
def _state(pg) -> dict: return json.loads(pg.evaluate("JSON.stringify(document.querySelector('[data-rupaka]').rupaka.st)"))

def _close(a, b):
    "Equal, with numbers compared to six digits."
    if isinstance(a, (int, float)) and isinstance(b, (int, float)) and not isinstance(a, bool): return math.isclose(a, b, rel_tol=1e-6, abs_tol=1e-9) or (a != a and b != b)
    if isinstance(a, dict) and isinstance(b, dict): return a.keys() == b.keys() and all(_close(a[k], b[k]) for k in a)
    if isinstance(a, list) and isinstance(b, list): return len(a) == len(b) and all(map(_close, a, b))
    return a == b

def _parity(pg, spec) -> list:
    "Errors for derived values where the runtime and the Python twin differ."
    st = _state(pg)
    py = run(spec, {k: st[k] for k in spec.get('state', {})})
    return [f'derive "{d["out"]}": the runtime gives {json.dumps(st.get(d["out"]))[:80]}, Python gives {json.dumps(py.get(d["out"]))[:80]}'
            for d in spec.get('derive') or [] if not _close(st.get(d['out']), py.get(d['out']))]

`_boxes` checks that each visible block has a box, and that each block with `after` is hidden before its gate.
Chrome's `checkVisibility()` does not see `display: none` on an SVG element, so `VIS` also looks at the element and its parents.

In [ ]:
#| export
VIS = "const vis = e => { if (!e.checkVisibility()) return false; for (let n = e; n && n.nodeType === 1; n = n.parentElement) if (getComputedStyle(n).display === 'none') return false; return true; };"

def _boxes(pg) -> list:
    r = pg.evaluate("() => { " + VIS + """ return [...document.querySelectorAll('.rk-b')].map(e => { const b = e.getBoundingClientRect();
        return {type: e.dataset.rkType, after: e.dataset.rkAfter || '', vis: vis(e), w: b.width, h: b.height}; }); }""")
    return ([f'{x["type"]}: the block is visible, but its box is empty' for x in r if x['vis'] and not x['after'] and (x['w'] < 1 or x['h'] < 1)] +
            [f'{x["type"]}: the block waits for gate "{x["after"]}", but it shows before the prediction' for x in r if x['after'] and x['vis']])

`_play` plays each gate: it picks the first option with the key 1, and locks in with ⌘↩.

In [ ]:
#| export
def _play(pg, spec) -> tuple:
    errs, info = [], []
    pg.evaluate("window.__rk = []; document.addEventListener('rk-predict', e => window.__rk.push(e.detail))")
    for g in _gates(spec['blocks']):
        loc = pg.locator('.rk-gate').nth(len(info))
        if not loc.is_visible(): errs.append(f'gate "{g["id"]}": it is hidden, so the learner cannot reach it'); info.append(None); continue
        st = _state(pg)
        py = run(spec, {k: st[k] for k in spec.get('state', {})})
        want = g['answer']['is'] if isinstance(g['answer'], dict) else get(py, g['answer'])
        opts = loc.locator('.opts button')
        if not opts.count(): errs.append(f'gate "{g["id"]}": it has no options'); info.append(None); continue
        texts = opts.evaluate_all("bs => bs.map(b => b.firstChild.textContent)")
        opts.first.focus(); pg.keyboard.press('1'); pg.keyboard.press('Meta+Enter')
        ev = [e for e in pg.evaluate('window.__rk') if e['gate'] == g['id']]
        verdict = loc.locator('.verdict')
        if not ev: errs.append(f'gate "{g["id"]}": ⌘↩ did not lock in'); info.append(None); continue
        if not verdict.is_visible() or 'You said' not in verdict.inner_text(): errs.append(f'gate "{g["id"]}": no verdict after the lock in')
        if js_str(ev[-1]['ans']) != js_str(want) and not _close(ev[-1]['ans'], want):
            errs.append(f'gate "{g["id"]}": the runtime answer is {js_str(ev[-1]["ans"])}, Python gives {js_str(want)}')
        hid = pg.evaluate("() => { " + VIS + f"""return [...document.querySelectorAll('[data-rk-after="{g["id"]}"]')].filter(e => !vis(e)).map(e => e.dataset.rkType); }}""")
        errs += [f'gate "{g["id"]}": the {t} that waits for it did not show' for t in hid]
        info.append(dict(gate=g['id'], options=texts, picked=ev[-1]['picked'], ans=ev[-1]['ans'], ok=ev[-1]['ok'], verdict=verdict.inner_text()))
    return errs, info

`_keys` sends keys to each control and checks that the state moves. A control at one end of its range gets the opposite key too.

In [ ]:
#| export
CONTROLS = {'slider': ('input', ['ArrowRight', 'ArrowLeft']), 'toggle': ('button', ['Space']), 'choice': ('.opts button[tabindex="0"]', ['ArrowRight', 'ArrowLeft']),
            'wheel': ('svg', ['ArrowRight']), 'ranklist': ('g.row', ['Alt+ArrowDown']), 'grid': ('svg', ['+']), 'point': (None, ['ArrowRight', 'ArrowLeft'])}

def _inputs(pg, spec): st = _state(pg); return {k: st.get(k) for k in spec.get('state', {})}

def _keys(pg, spec) -> list:
    errs = []
    for t, (sel, keys) in CONTROLS.items():
        locs = pg.locator(f'.rk-{t}.drag' if t == 'point' else f'[data-rk-type="{t}"]')
        for j in range(locs.count()):
            loc = locs.nth(j)
            if not loc.is_visible(): continue
            tgt = loc.locator(sel).first if sel else loc
            before = _inputs(pg, spec)
            for k in keys:
                tgt.focus(); pg.keyboard.press(k)
                if _inputs(pg, spec) != before: break
            else: errs.append(f'{t} {j + 1}: the keys {", ".join(keys)} did not change the state')
    return errs

## check

`check` puts the parts together. It returns `AttrDict(ok, errors, info, shots)`. With `shots`, it saves screenshots at 2× before and after the play, in light and dark.

In [ ]:
#| export
def _check(b, spec, shots=None, modes=('light', 'dark'), play=True, width=720):
    errs, info, files = [], {}, []
    p = save(spec, Path(tempfile.mkdtemp())/'demo.html')
    for mode in modes:
        pg = b.new_page(viewport=dict(width=width, height=900), color_scheme=mode, device_scale_factor=2 if shots else 1)
        log = []
        pg.on('console', lambda m: m.type == 'error' and log.append(f'console: {m.text}'))
        pg.on('pageerror', lambda e: log.append(f'page error: {e}'))
        pg.on('request', lambda r: r.url.split(':')[0] not in ('file', 'data', 'about') and log.append(f'network: {r.url}'))
        pg.goto(p.as_uri())
        try: pg.wait_for_selector('[data-rk-ready]', state='attached', timeout=5000)
        except Exception: log.append('the demo did not draw in 5 s')
        ready = pg.evaluate("document.querySelector('[data-rupaka]')?.dataset.rkReady")
        e = []
        if ready != '1':
            e.append(pg.locator('.rk-err').inner_text() if pg.locator('.rk-err').count() else 'the demo did not draw')
        else:
            e += _boxes(pg) + _parity(pg, spec) + [f'contrast: "{t}" is {r}:1, it needs {n}:1' for t, r, n in contrast(pg)]
            if shots: files.append(_shot(pg, shots, spec, mode, 'before'))
            if play:
                pe, info[mode] = _play(pg, spec)
                e += pe + [f'contrast after the reveal: "{t}" is {r}:1, it needs {n}:1' for t, r, n in contrast(pg)]
                e += _keys(pg, spec) + _parity(pg, spec)
            info[f'{mode}_height'] = pg.evaluate("document.querySelector('.rk').getBoundingClientRect().height")
            if shots: files.append(_shot(pg, shots, spec, mode, 'after'))
        errs += [f'{mode}: {x}' for x in dict.fromkeys(e + log)]
        pg.close()
    return AttrDict(ok=not errs, errors=errs, info=info, shots=files)

def _shot(pg, d, spec, mode, stage):
    pg.wait_for_timeout(350)   # let the bars slide into place
    f = Path(d)/f'{spec.get("id", "demo")}-{mode}-{stage}.png'
    f.parent.mkdir(parents=True, exist_ok=True)
    pg.screenshot(path=f, full_page=True)
    return f

`check` runs in a thread when an asyncio loop runs, as in a notebook.

In [ ]:
#| export
def check(spec, shots=None, modes=('light', 'dark'), play:bool=True, b=None, path:str=None) -> AttrDict:
    "Draw `spec` in a headless browser, play it, and return `AttrDict(ok, errors, info, shots)`; `b` reuses an open browser."
    if b is not None: return _check(b, spec, shots, modes, play)
    def go():
        with browser(path) as br: return _check(br, spec, shots, modes, play)
    return in_thread(go)

## Tests

These tests need a browser, so they run only where one starts.

In [ ]:
ok = can_check()
ok

A good demo passes. The info has the options of the gate and the verdict.

In [ ]:
from rupaka.spec import examples
if ok:
    r = check(examples()['gate'])
    test_eq(r.errors, [])
    test_eq(r.info['light'][0]['options'], ['4', '5', '6', '23'])
    assert r.info['light'][0]['verdict'].startswith('You said 4. It is 5.')

A spec with a broken fn name fails with the error text of the runtime.

In [ ]:
if ok:
    bad = {**examples()['gate'], 'derive': [{'fn': 'sums', 'out': 'sum'}]}
    r = check(bad, modes=('light',))
    assert not r.ok and 'unknown fn "sums"' in r.errors[0], r.errors

Each block example passes the whole check, in light and dark.

In [ ]:
def check_all(specs):
    with browser() as br: return {n: check(s, b=br).errors for n, s in specs.items()}
if ok: test_eq(in_thread(check_all, examples()), {n: [] for n in examples()})

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()